# Defeitos de fabricação

Exploração dos registros, das datas e dos custos de reparo.

In [8]:
from pathlib import Path

import pandas as pd
import plotly.express as px

arquivo = Path('defects_data.csv')

df = pd.read_csv(arquivo)
df['defect_date'] = pd.to_datetime(df['defect_date'], errors='coerce')

print(f'Arquivo: {arquivo} | Registros: {len(df):,} | Colunas: {df.shape[1]}')
display(df.head())

Arquivo: defects_data.csv | Registros: 1,000 | Colunas: 8


,defect_id,product_id,defect_type,defect_date,defect_location,severity,inspection_method,repair_cost
0,1,15,Structural,2024-06-06,Component,Minor,Visual Inspection,245.47
1,2,6,Functional,2024-04-26,Component,Minor,Visual Inspection,26.87
2,3,84,Structural,2024-02-15,Internal,Minor,Automated Testing,835.81
3,4,10,Functional,2024-03-28,Internal,Critical,Automated Testing,444.47
4,5,14,Cosmetic,2024-04-26,Component,Minor,Manual Testing,823.64


In [9]:
# Estrutura e qualidade dos dados
resumo = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'preenchidos': df.notna().sum(),
    'ausentes': df.isna().sum(),
    'ausentes (%)': (df.isna().mean() * 100).round(1),
    'valores distintos': df.nunique(dropna=True),
})
display(resumo)
print(f'Linhas duplicadas: {df.duplicated().sum():,}')
print(f"Datas inválidas: {df['defect_date'].isna().sum():,}")

if df['defect_date'].notna().any():
    print(f"Período: {df['defect_date'].min():%d/%m/%Y} a {df['defect_date'].max():%d/%m/%Y}")
if 'repair_cost' in df:
    display(df['repair_cost'].describe().to_frame('Custo de reparo').round(2))

,tipo,preenchidos,ausentes,ausentes (%),valores distintos
defect_id,int64,1000,0,0.0,1000
product_id,int64,1000,0,0.0,100
defect_type,str,1000,0,0.0,3
defect_date,datetime64[us],1000,0,0.0,182
defect_location,str,1000,0,0.0,3
severity,str,1000,0,0.0,3
inspection_method,str,1000,0,0.0,3
repair_cost,float64,1000,0,0.0,996


Linhas duplicadas: 0
Datas inválidas: 0
Período: 01/01/2024 a 30/06/2024


,Custo de reparo
count,1000.00
mean,507.63
std,289.62
min,10.22
25%,270.90
50%,506.43
75%,759.06
max,999.64


In [10]:
# Frequência e participação de cada categoria
categorias = ['defect_type', 'defect_location', 'severity', 'inspection_method']
for coluna in categorias:
    contagem = df[coluna].value_counts(dropna=False).rename_axis(coluna).reset_index(name='registros')
    contagem['participacao (%)'] = (100 * contagem['registros'] / len(df)).round(1)
    display(contagem)

    fig = px.bar(
        contagem.sort_values('registros'), x='registros', y=coluna,
        orientation='h', text='registros',
        title=f'Registros por {coluna.replace("_", " ")}',
        labels={'registros': 'Registros'},
    )
    fig.update_layout(showlegend=False)
    fig.show()

produtos = df['product_id'].value_counts().head(10).sort_values().rename_axis('product_id').reset_index(name='registros')
px.bar(
    produtos, x='registros', y='product_id', orientation='h', text='registros',
    title='Produtos com mais registros de defeito (10 primeiros)',
    labels={'registros': 'Registros', 'product_id': 'Produto'},
).show()

,defect_type,registros,participacao (%)
0,Structural,352,35.2
1,Functional,339,33.9
2,Cosmetic,309,30.9


,defect_location,registros,participacao (%)
0,Surface,353,35.3
1,Component,326,32.6
2,Internal,321,32.1


,severity,registros,participacao (%)
0,Minor,358,35.8
1,Critical,333,33.3
2,Moderate,309,30.9


,inspection_method,registros,participacao (%)
0,Manual Testing,352,35.2
1,Visual Inspection,351,35.1
2,Automated Testing,297,29.7


In [11]:
# Evolução mensal: meses sem registros também aparecem no intervalo
com_data = df.dropna(subset=['defect_date'])
if not com_data.empty:
    mensal = (
        com_data.set_index('defect_date').resample('MS').size()
        .rename('registros').reset_index()
    )
    px.line(
        mensal, x='defect_date', y='registros', markers=True,
        title='Registros de defeito por mês',
        labels={'defect_date': 'Mês', 'registros': 'Registros'},
    ).show()

    por_ano_mes = com_data.groupby(com_data['defect_date'].dt.to_period('M')).size()
    print(f'Mês com mais registros: {por_ano_mes.idxmax()} ({por_ano_mes.max()} registros)')
else:
    print('Não há datas válidas para analisar a evolução mensal.')

Mês com mais registros: 2024-01 (191 registros)


In [12]:
# Custos: distribuição geral e comparação entre grupos
px.histogram(
    df, x='repair_cost', marginal='box', nbins=30,
    title='Distribuição dos custos de reparo',
    labels={'repair_cost': 'Custo de reparo'},
).show()

for coluna in ['severity', 'defect_type']:
    fig = px.box(
        df, x=coluna, y='repair_cost', points='outliers',
        title=f'Custo de reparo por {coluna.replace("_", " ")}',
        labels={coluna: coluna.replace('_', ' ').capitalize(), 'repair_cost': 'Custo de reparo'},
    )
    fig.update_xaxes(categoryorder='median descending')
    fig.show()

custos = df.groupby('severity', dropna=False)['repair_cost'].agg(
    registros='count', mediana='median', media='mean', minimo='min', maximo='max'
).sort_values('mediana', ascending=False)
display(custos.round(2))

,registros,mediana,media,minimo,maximo
severity,,,,,
Minor,358,508.37,514.43,10.44,998.27
Critical,333,503.25,505.87,10.22,999.25
Moderate,309,498.71,501.63,10.55,999.64


In [13]:
# Composição da severidade dentro de cada tipo de defeito
composicao = pd.crosstab(df['defect_type'], df['severity'], normalize='index')
if not composicao.empty:
    fig = px.imshow(
        composicao, aspect='auto', color_continuous_scale='Blues',
        labels={'x': 'Severidade', 'y': 'Tipo de defeito', 'color': 'Proporção'},
        title='Severidade registrada por tipo de defeito (percentual por linha)',
    )
    fig.update_coloraxes(colorbar_tickformat='.0%')
    fig.show()
    display((composicao * 100).round(1).rename_axis('Tipo de defeito'))

severity,Critical,Minor,Moderate
Tipo de defeito,,,
Cosmetic,34.6,34.3,31.1
Functional,33.6,33.3,33.0
Structural,31.8,39.5,28.7


In [14]:
# Dispersao entre variaveis numericas
variaveis_numericas = [
    coluna for coluna in df.select_dtypes(include='number').columns
    if coluna != 'defect_id'
]

if len(variaveis_numericas) >= 2:
    fig = px.scatter_matrix(
        df,
        dimensions=variaveis_numericas,
        color='severity' if 'severity' in df.columns else None,
        title='Dispersao entre variaveis numericas',
        hover_data=['defect_type'] if 'defect_type' in df.columns else None,
    )
    fig.update_traces(diagonal_visible=False, showupperhalf=False, marker={'opacity': 0.65, 'size': 6})
    fig.show()

if {'defect_date', 'repair_cost'}.issubset(df.columns):
    px.scatter(
        df,
        x='defect_date',
        y='repair_cost',
        color='severity' if 'severity' in df.columns else None,
        hover_data=[coluna for coluna in ('product_id', 'defect_type') if coluna in df.columns],
        title='Custo de reparo por data do defeito',
        labels={'defect_date': 'Data do defeito', 'repair_cost': 'Custo de reparo'},
    ).show()

# Registros por mês, segmentados por categoria
dados_mensais = df.dropna(subset=['defect_date']).copy()
dados_mensais['mes'] = dados_mensais['defect_date'].dt.to_period('M').dt.to_timestamp()
meses = pd.date_range(dados_mensais['mes'].min(), dados_mensais['mes'].max(), freq='MS')

for categoria in categorias:
    valores = dados_mensais[categoria].dropna().unique()
    indice_completo = pd.MultiIndex.from_product(
        [meses, valores], names=['mes', categoria]
    )
    contagem_mensal = (
        dados_mensais.dropna(subset=[categoria])
        .groupby(['mes', categoria])
        .size()
        .reindex(indice_completo, fill_value=0)
        .rename('registros')
        .reset_index()
    )
    fig = px.bar(
        contagem_mensal,
        x='mes',
        y='registros',
        color=categoria,
        barmode='group',
        title=f'Registros por mês e {categoria.replace("_", " ")}',
        labels={'mes': 'Mês', 'registros': 'Registros', categoria: categoria.replace('_', ' ').capitalize()},
    )
    fig.update_xaxes(tickformat='%b/%Y')
    fig.show()